# Zava Chapters 01 and 01a — Interactive Azure Deployment

Run this Python Jupyter notebook **one cell at a time, from top to bottom**. It deploys the existing Zava Bicep stages while preserving validation and approval gates.

## Safety rules

- Lab-only VNet/DNS simulation templates are intentionally excluded.
- Every Azure-changing cell requires an approval variable to be changed from `False` to `True`.
- Review each `what-if` output before deployment.
- Confirm in writing that `dmzsubnet-1` is `10.75.139.128/29`; the originally supplied `10.75.128/29` is outside the VNet.
- Capability hosts are one-time resources. Never blindly rerun their deployment.
- Run private DNS checks from a Zava-connected machine using the approved DNS path.

The notebook uses Azure SDKs for authentication/resource inspection and Azure CLI argument arrays for Bicep deployment operations. It never invokes a shell.

## 1. Install and import Azure SDK packages

Run the next cell once for the selected Python kernel. Restart the kernel only if VS Code requests it.

In [ ]:
%pip install --quiet azure-identity azure-mgmt-resource azure-mgmt-subscription pandas

import ipaddress
import json
import socket
import subprocess
import tempfile
from pathlib import Path

import pandas as pd
from azure.identity import AzureCliCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.subscription import SubscriptionClient
from IPython.display import Markdown, display

print("Imports completed.")

## 2. Define deployment configuration

Replace all placeholders. Choose the DNS ownership mode before deployment.

In [ ]:
TENANT_ID = "<Zava-tenant-id>"
SUBSCRIPTION = "<Zava-subscription-id-or-name>"
WORKLOAD_RG = "<Zava-workload-resource-group>"
NETWORK_RG = "<Zava-network-resource-group>"
DNS_RG = "<Zava-private-dns-resource-group>"
LOCATION = "eastus"
SEARCH_LOCATION = "eastus"
VNET_NAME = "azr-133-eastus"
AGENT_SUBNET_NAME = "snet-zava-foundry-agent"
AGENT_SUBNET_PREFIX = "10.75.139.160/27"
PE_SUBNET_NAME = "snet-zava-privateendpoints"
PE_SUBNET_PREFIX = "10.75.139.144/28"
WORKLOAD_NAME = "zava"
ENVIRONMENT = "dev"

# True: notebook creates PE DNS-zone groups referencing existing Zava zones.
# False: Zava DNS automation creates the zone groups/records after PE creation.
CREATE_PE_DNS_ZONE_GROUPS = True

# Leave blank to grant the signed-in user; otherwise supply an approved Entra user object ID.
FOUNDRY_USER_OBJECT_ID = ""

# Find the repository from the current working directory or its parents.
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Run this notebook from the cloned Git repository.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
INFRA_DIR = REPO_ROOT / "infra" / "zava-chapter-01-01a-step-by-step"
SUBNET_TEMPLATE = INFRA_DIR / "01-create-foundry-subnets.bicep"
DNS_LINK_TEMPLATE = INFRA_DIR / "02-link-customer-private-dns.bicep"
CHAPTER01_TEMPLATE = INFRA_DIR / "03-chapter-01-foundation.bicep"
CORE_TEMPLATE = INFRA_DIR / "04-chapter-01a-standard-core.bicep"
HOST_TEMPLATE = INFRA_DIR / "05-chapter-01a-capability-hosts.bicep"

for name, value in {
    "TENANT_ID": TENANT_ID,
    "SUBSCRIPTION": SUBSCRIPTION,
    "WORKLOAD_RG": WORKLOAD_RG,
    "NETWORK_RG": NETWORK_RG,
    "DNS_RG": DNS_RG,
}.items():
    if not value or (value.startswith("<") and value.endswith(">")):
        raise ValueError(f"Configure {name} before continuing.")

print(json.dumps({
    "repository": str(REPO_ROOT),
    "subscription": SUBSCRIPTION,
    "tenant_id": TENANT_ID,
    "workload_rg": WORKLOAD_RG,
    "network_rg": NETWORK_RG,
    "dns_rg": DNS_RG,
    "vnet": VNET_NAME,
    "agent_subnet": f"{AGENT_SUBNET_NAME} ({AGENT_SUBNET_PREFIX})",
    "private_endpoint_subnet": f"{PE_SUBNET_NAME} ({PE_SUBNET_PREFIX})",
    "create_pe_dns_zone_groups": CREATE_PE_DNS_ZONE_GROUPS,
}, indent=2))

In [ ]:
def run_command(args, *, capture=False, check=True, input_text=None):
    """Run a command safely with an argument list; never invokes a shell."""
    printable = " ".join(str(arg) for arg in args)
    print(f"> {printable}")
    result = subprocess.run(
        [str(arg) for arg in args],
        check=False,
        text=True,
        input=input_text,
        capture_output=capture,
    )
    if capture and result.stdout:
        print(result.stdout.rstrip())
    if capture and result.stderr:
        print(result.stderr.rstrip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {printable}\n{result.stderr}")
    return result


def az(*args, capture=False, check=True):
    return run_command(["az", *args], capture=capture, check=check)


def az_json(*args):
    result = az(*args, "--output", "json", capture=True)
    return json.loads(result.stdout)


def build_bicep(template: Path):
    az("bicep", "build", "--file", str(template), "--stdout", capture=True)
    print(f"Bicep compilation passed: {template.name}")


def deploy_args(resource_group, name, template, parameters):
    args = [
        "deployment", "group",
        "--resource-group", resource_group,
        "--name", name,
        "--template-file", str(template),
        "--parameters", *parameters,
    ]
    return args


def require_approval(approved: bool, action: str):
    if not approved:
        raise RuntimeError(f"{action} blocked. Review what-if, then set the approval variable to True.")


def show_source(path: Path):
    display(Markdown(f"### `{path.name}`"))
    print(path.read_text(encoding="utf-8"))

print("Helper functions loaded.")

## 3. Authenticate with Azure

The Azure CLI login is used by both the SDK credential and Bicep commands. Verify the immutable tenant and subscription IDs before continuing.

In [ ]:
az("login", "--tenant", TENANT_ID, "--use-device-code")
az("account", "set", "--subscription", SUBSCRIPTION)
account = az_json("account", "show")
if account["tenantId"] != TENANT_ID:
    raise RuntimeError(f"Tenant mismatch: active={account['tenantId']} expected={TENANT_ID}")

SUBSCRIPTION_ID = account["id"]
if not FOUNDRY_USER_OBJECT_ID:
    user_result = az("ad", "signed-in-user", "show", "--query", "id", "--output", "tsv", capture=True)
    FOUNDRY_USER_OBJECT_ID = user_result.stdout.strip()

credential = AzureCliCredential(tenant_id=TENANT_ID)
resource_client = ResourceManagementClient(credential, SUBSCRIPTION_ID)
subscription_client = SubscriptionClient(credential)

print(json.dumps({
    "subscription": account["name"],
    "subscription_id": SUBSCRIPTION_ID,
    "tenant_id": account["tenantId"],
    "user": account["user"]["name"],
    "foundry_user_object_id": FOUNDRY_USER_OBJECT_ID,
}, indent=2))

## 4. Validate subscription, providers, and resource groups

Provider registration remains a Zava subscription-team responsibility. The notebook only reports missing registrations.

In [ ]:
REQUIRED_PROVIDERS = [
    "Microsoft.App", "Microsoft.CognitiveServices", "Microsoft.ContainerService",
    "Microsoft.DocumentDB", "Microsoft.KeyVault", "Microsoft.Network",
    "Microsoft.Search", "Microsoft.Storage",
]
provider_rows = []
for namespace in REQUIRED_PROVIDERS:
    provider = resource_client.providers.get(namespace)
    provider_rows.append({"provider": namespace, "state": provider.registration_state})
display(pd.DataFrame(provider_rows))
missing = [row["provider"] for row in provider_rows if row["state"] != "Registered"]
if missing:
    raise RuntimeError(f"Zava must register these providers: {', '.join(missing)}")

rg_rows = []
for rg_name in {WORKLOAD_RG, NETWORK_RG, DNS_RG}:
    rg_rows.append({"resource_group": rg_name, "exists": resource_client.resource_groups.check_existence(rg_name)})
display(pd.DataFrame(rg_rows))

for required_rg in {NETWORK_RG, DNS_RG}:
    if not resource_client.resource_groups.check_existence(required_rg):
        raise RuntimeError(f"Required customer-owned resource group does not exist: {required_rg}")

if not resource_client.resource_groups.check_existence(WORKLOAD_RG):
    print("Workload resource group is missing. Use the next guarded cell only if Zava approves its creation.")

## 5. Optionally create or update the workload resource group

Normally Zava provides this resource group in advance. If it does not exist and the customer has approved its creation, the next guarded cell creates it through the Azure Resource Management SDK. It never creates or changes the network or DNS resource groups.

In [ ]:
APPROVE_WORKLOAD_RG_CREATE_OR_UPDATE = False

workload_rg_exists = resource_client.resource_groups.check_existence(WORKLOAD_RG)
if workload_rg_exists:
    workload_rg = resource_client.resource_groups.get(WORKLOAD_RG)
    print(f"Workload resource group already exists: {workload_rg.id}")
else:
    require_approval(APPROVE_WORKLOAD_RG_CREATE_OR_UPDATE, "Workload resource-group creation")
    workload_rg = resource_client.resource_groups.create_or_update(
        WORKLOAD_RG,
        {
            "location": LOCATION,
            "tags": {
                "customer": "Zava",
                "workload": WORKLOAD_NAME,
                "environment": ENVIRONMENT,
                "managedBy": "Jupyter-Bicep",
            },
        },
    )
    print(f"Created workload resource group: {workload_rg.id}")

## 5. Inspect and prepare the customer VNet

The next cells inspect `azr-133-eastus`, display the exact subnet Bicep, validate it, preview it, and optionally deploy it. **Written confirmation of the corrected DMZ range is mandatory.**

In [ ]:
vnet = az_json("network", "vnet", "show", "-g", NETWORK_RG, "-n", VNET_NAME)
subnets = az_json("network", "vnet", "subnet", "list", "-g", NETWORK_RG, "--vnet-name", VNET_NAME)
print(json.dumps({
    "name": vnet["name"],
    "location": vnet["location"],
    "address_space": vnet["addressSpace"]["addressPrefixes"],
    "dns_servers": vnet.get("dhcpOptions", {}).get("dnsServers", []),
}, indent=2))
display(pd.DataFrame([
    {"name": s["name"], "prefix": s.get("addressPrefix"), "state": s.get("provisioningState")}
    for s in subnets
]))

DMZ_RANGE_CONFIRMED_IN_WRITING = False
if not DMZ_RANGE_CONFIRMED_IN_WRITING:
    raise RuntimeError("Obtain written Zava confirmation that dmzsubnet-1 is 10.75.139.128/29, then set the gate to True.")

In [ ]:
show_source(SUBNET_TEMPLATE)
build_bicep(SUBNET_TEMPLATE)
SUBNET_DEPLOYMENT = "zava-01-foundry-subnets"
subnet_parameters = [
    f"vnetName={VNET_NAME}", f"agentSubnetName={AGENT_SUBNET_NAME}",
    f"agentSubnetPrefix={AGENT_SUBNET_PREFIX}", f"privateEndpointSubnetName={PE_SUBNET_NAME}",
    f"privateEndpointSubnetPrefix={PE_SUBNET_PREFIX}",
]
az("deployment", "group", "validate", "-g", NETWORK_RG, "-n", f"{SUBNET_DEPLOYMENT}-validate",
   "-f", str(SUBNET_TEMPLATE), "--parameters", *subnet_parameters, "-o", "none")
az("deployment", "group", "what-if", "-g", NETWORK_RG, "-n", SUBNET_DEPLOYMENT,
   "-f", str(SUBNET_TEMPLATE), "--parameters", *subnet_parameters, "--result-format", "ResourceIdOnly")

In [ ]:
APPROVE_SUBNET_DEPLOYMENT = False
require_approval(APPROVE_SUBNET_DEPLOYMENT, "Subnet deployment")
az("deployment", "group", "create", "-g", NETWORK_RG, "-n", SUBNET_DEPLOYMENT,
   "-f", str(SUBNET_TEMPLATE), "-p", *subnet_parameters, "-o", "none")
print("Subnet deployment completed.")

In [ ]:
subnets = az_json("network", "vnet", "subnet", "list", "-g", NETWORK_RG, "--vnet-name", VNET_NAME)
display(pd.DataFrame([{
    "name": s["name"], "prefix": s.get("addressPrefix"),
    "delegation": ",".join(d["serviceName"] for d in s.get("delegations", [])),
    "pe_policies": s.get("privateEndpointNetworkPolicies"), "state": s.get("provisioningState")
} for s in subnets]))
agent_subnet = az_json("network", "vnet", "subnet", "show", "-g", NETWORK_RG, "--vnet-name", VNET_NAME, "-n", AGENT_SUBNET_NAME)
AGENT_SUBNET_ID = agent_subnet["id"]
print(f"Agent subnet ID: {AGENT_SUBNET_ID}")

## 6. Validate customer-owned private DNS

Choose one topology:

- **Direct spoke links:** preview and deploy the link template below.
- **Central hub resolver:** skip link deployment; Zava's resolver must provide private resolution.

Do not create duplicate private DNS zones in the workload resource group.

In [ ]:
DNS_ZONES = [
    "privatelink.cognitiveservices.azure.com", "privatelink.openai.azure.com",
    "privatelink.services.ai.azure.com", "privatelink.blob.core.windows.net",
    "privatelink.vaultcore.azure.net", "privatelink.documents.azure.com",
    "privatelink.search.windows.net",
]
zone_rows = []
for zone in DNS_ZONES:
    result = az("network", "private-dns", "zone", "show", "-g", DNS_RG, "-n", zone,
                "--query", "{name:name,id:id}", "-o", "json", capture=True, check=False)
    zone_rows.append({"zone": zone, "exists": result.returncode == 0})
display(pd.DataFrame(zone_rows))
missing_zones = [row["zone"] for row in zone_rows if not row["exists"]]
if missing_zones:
    raise RuntimeError(f"Missing Zava DNS zones: {', '.join(missing_zones)}")

In [ ]:
# OPTIONAL — direct spoke-link topology only.
show_source(DNS_LINK_TEMPLATE)
build_bicep(DNS_LINK_TEMPLATE)
DNS_LINK_DEPLOYMENT = "zava-02-private-dns-links"
dns_link_parameters = [f"networkResourceGroupName={NETWORK_RG}", f"vnetName={VNET_NAME}"]
az("deployment", "group", "validate", "-g", DNS_RG, "-n", f"{DNS_LINK_DEPLOYMENT}-validate",
   "-f", str(DNS_LINK_TEMPLATE), "-p", *dns_link_parameters, "-o", "none")
az("deployment", "group", "what-if", "-g", DNS_RG, "-n", DNS_LINK_DEPLOYMENT,
   "-f", str(DNS_LINK_TEMPLATE), "-p", *dns_link_parameters, "--result-format", "ResourceIdOnly")

In [ ]:
APPROVE_DIRECT_DNS_LINKS = False
require_approval(APPROVE_DIRECT_DNS_LINKS, "Direct DNS-link deployment")
az("deployment", "group", "create", "-g", DNS_RG, "-n", DNS_LINK_DEPLOYMENT,
   "-f", str(DNS_LINK_TEMPLATE), "-p", *dns_link_parameters, "-o", "none")
for zone in DNS_ZONES:
    az("network", "private-dns", "link", "vnet", "list", "-g", DNS_RG, "-z", zone,
       "--query", f"[?contains(virtualNetwork.id, '/virtualNetworks/{VNET_NAME}')].{{name:name,state:virtualNetworkLinkState}}", "-o", "table")

## 7. Deploy Chapter 01 foundation

Creates the private Basic Foundry account/project, Storage, Key Vault, and three private endpoints. No model is deployed.

In [ ]:
show_source(CHAPTER01_TEMPLATE)
build_bicep(CHAPTER01_TEMPLATE)
CHAPTER01_DEPLOYMENT = f"zava-03-chapter-01-{ENVIRONMENT}"
chapter01_parameters = [
    f"location={LOCATION}", f"workloadName={WORKLOAD_NAME}", f"environment={ENVIRONMENT}",
    f"networkResourceGroupName={NETWORK_RG}", f"vnetName={VNET_NAME}",
    f"privateEndpointSubnetName={PE_SUBNET_NAME}", f"privateDnsResourceGroupName={DNS_RG}",
    f"createPrivateEndpointDnsZoneGroups={str(CREATE_PE_DNS_ZONE_GROUPS).lower()}",
    f"foundryUserPrincipalId={FOUNDRY_USER_OBJECT_ID}",
]
az("deployment", "group", "validate", "-g", WORKLOAD_RG, "-n", f"{CHAPTER01_DEPLOYMENT}-validate",
   "-f", str(CHAPTER01_TEMPLATE), "-p", *chapter01_parameters, "-o", "none")
az("deployment", "group", "what-if", "-g", WORKLOAD_RG, "-n", CHAPTER01_DEPLOYMENT,
   "-f", str(CHAPTER01_TEMPLATE), "-p", *chapter01_parameters, "--result-format", "ResourceIdOnly")

In [ ]:
APPROVE_CHAPTER01_DEPLOYMENT = False
require_approval(APPROVE_CHAPTER01_DEPLOYMENT, "Chapter 01 deployment")
az("deployment", "group", "create", "-g", WORKLOAD_RG, "-n", CHAPTER01_DEPLOYMENT,
   "-f", str(CHAPTER01_TEMPLATE), "-p", *chapter01_parameters, "-o", "none")
print("Chapter 01 deployment completed.")

In [ ]:
chapter01 = az_json("deployment", "group", "show", "-g", WORKLOAD_RG, "-n", CHAPTER01_DEPLOYMENT)
chapter01_outputs = {k: v["value"] for k, v in chapter01["properties"]["outputs"].items()}
BASIC_FOUNDRY = chapter01_outputs["foundryAccountName"]
BASIC_PROJECT = chapter01_outputs["foundryProjectName"]
BASIC_STORAGE = chapter01_outputs["storageAccountName"]
BASIC_VAULT = chapter01_outputs["keyVaultName"]
print(json.dumps(chapter01_outputs, indent=2))
az("cognitiveservices", "account", "show", "-g", WORKLOAD_RG, "-n", BASIC_FOUNDRY,
   "--query", "{name:name,state:properties.provisioningState,public:properties.publicNetworkAccess,localAuthDisabled:properties.disableLocalAuth}", "-o", "table")
az("storage", "account", "show", "-g", WORKLOAD_RG, "-n", BASIC_STORAGE,
   "--query", "{name:name,public:publicNetworkAccess,sharedKey:allowSharedKeyAccess}", "-o", "table")
az("keyvault", "show", "-g", WORKLOAD_RG, "-n", BASIC_VAULT,
   "--query", "{name:name,public:properties.publicNetworkAccess,rbac:properties.enableRbacAuthorization,purge:properties.enablePurgeProtection}", "-o", "table")

## 8. Deploy Chapter 01a Standard core

Creates network-injected Standard Foundry, private Storage, Cosmos DB, Search, four private endpoints, managed-identity roles, and three AAD project connections. Capability hosts and the model remain separate.

In [ ]:
show_source(CORE_TEMPLATE)
build_bicep(CORE_TEMPLATE)
CORE_DEPLOYMENT = f"zava-04-chapter-01a-core-{ENVIRONMENT}"
core_parameters = [
    f"location={LOCATION}", f"searchLocation={SEARCH_LOCATION}",
    f"workloadName={WORKLOAD_NAME}", f"environment={ENVIRONMENT}",
    f"networkResourceGroupName={NETWORK_RG}", f"vnetName={VNET_NAME}",
    f"agentSubnetName={AGENT_SUBNET_NAME}", f"privateEndpointSubnetName={PE_SUBNET_NAME}",
    f"privateDnsResourceGroupName={DNS_RG}",
    f"createPrivateEndpointDnsZoneGroups={str(CREATE_PE_DNS_ZONE_GROUPS).lower()}",
    f"foundryUserPrincipalId={FOUNDRY_USER_OBJECT_ID}",
]
az("deployment", "group", "validate", "-g", WORKLOAD_RG, "-n", f"{CORE_DEPLOYMENT}-validate",
   "-f", str(CORE_TEMPLATE), "-p", *core_parameters, "-o", "none")
az("deployment", "group", "what-if", "-g", WORKLOAD_RG, "-n", CORE_DEPLOYMENT,
   "-f", str(CORE_TEMPLATE), "-p", *core_parameters, "--result-format", "ResourceIdOnly")

In [ ]:
APPROVE_CHAPTER01A_CORE = False
require_approval(APPROVE_CHAPTER01A_CORE, "Chapter 01a core deployment")
az("deployment", "group", "create", "-g", WORKLOAD_RG, "-n", CORE_DEPLOYMENT,
   "-f", str(CORE_TEMPLATE), "-p", *core_parameters, "-o", "none")
print("Chapter 01a core deployment completed.")

### Monitor deployment status and inspect errors

Azure CLI deployment creation waits for completion. This cell independently reads the ARM deployment state and prints failed deployment operations for diagnosis.

In [ ]:
deployment = resource_client.deployments.get(WORKLOAD_RG, CORE_DEPLOYMENT)
state = deployment.properties.provisioning_state
print(f"{CORE_DEPLOYMENT}: {state}")

if state == "Failed":
    operations = resource_client.deployment_operations.list(WORKLOAD_RG, CORE_DEPLOYMENT)
    failed_operations = []
    for operation in operations:
        operation_state = operation.properties.provisioning_state
        if operation_state == "Failed":
            failed_operations.append({
                "operation_id": operation.operation_id,
                "target": getattr(operation.properties.target_resource, "resource_name", None),
                "type": getattr(operation.properties.target_resource, "resource_type", None),
                "status_message": str(operation.properties.status_message),
            })
    display(pd.DataFrame(failed_operations))
    raise RuntimeError("Chapter 01a deployment failed. Review the failed operations above before continuing.")

In [ ]:
core = az_json("deployment", "group", "show", "-g", WORKLOAD_RG, "-n", CORE_DEPLOYMENT)
core_outputs = {k: v["value"] for k, v in core["properties"]["outputs"].items()}
STD_FOUNDRY = core_outputs["foundryAccountName"]
STD_PROJECT = core_outputs["foundryProjectName"]
STD_STORAGE = core_outputs["storageAccountName"]
COSMOS = core_outputs["cosmosAccountName"]
SEARCH = core_outputs["searchServiceName"]
PROJECT_MI = core_outputs["projectPrincipalId"]
STD_ID = core_outputs["foundryAccountId"]
print(json.dumps(core_outputs, indent=2))

injection = az_json("rest", "--method", "GET", "--url", f"https://management.azure.com{STD_ID}?api-version=2025-06-01")
print("Network injection:", json.dumps(injection["properties"].get("networkInjections"), indent=2))
az("cognitiveservices", "account", "show", "-g", WORKLOAD_RG, "-n", STD_FOUNDRY,
   "--query", "{state:properties.provisioningState,public:properties.publicNetworkAccess,localAuthDisabled:properties.disableLocalAuth}", "-o", "table")
az("storage", "account", "show", "-g", WORKLOAD_RG, "-n", STD_STORAGE,
   "--query", "{state:provisioningState,public:publicNetworkAccess,sharedKey:allowSharedKeyAccess}", "-o", "table")
az("cosmosdb", "show", "-g", WORKLOAD_RG, "-n", COSMOS,
   "--query", "{state:provisioningState,public:publicNetworkAccess,localAuthDisabled:disableLocalAuth}", "-o", "table")
az("search", "service", "show", "-g", WORKLOAD_RG, "-n", SEARCH,
   "--query", "{state:status,public:publicNetworkAccess,localAuthDisabled:disableLocalAuth,sku:sku.name}", "-o", "table")

In [ ]:
az("network", "private-endpoint", "list", "-g", WORKLOAD_RG,
   "--query", "[].{name:name,state:privateLinkServiceConnections[0].privateLinkServiceConnectionState.status,subnet:subnet.id}", "-o", "table")
az("role", "assignment", "list", "--assignee", PROJECT_MI, "--all",
   "--query", "[].{role:roleDefinitionName,scope:scope}", "-o", "table")
az("cosmosdb", "sql", "role", "assignment", "list", "-g", WORKLOAD_RG, "-a", COSMOS,
   "--query", f"[?principalId=='{PROJECT_MI}'].{{principalId:principalId,roleDefinitionId:roleDefinitionId,scope:scope}}", "-o", "table")
az("rest", "--method", "GET",
   "--url", f"https://management.azure.com{STD_ID}/projects/{STD_PROJECT}/connections?api-version=2025-06-01",
   "--query", "value[].{name:name,category:properties.category,auth:properties.authType,target:properties.target}", "-o", "table")

## 9. Mandatory private DNS checkpoint

Run this from a Zava-connected machine. Every resolved address must be inside `10.75.139.144/28`. Stop before capability hosts if any lookup fails or resolves publicly.

In [ ]:
private_endpoint_network = ipaddress.ip_network(PE_SUBNET_PREFIX)
names = [
    f"{STD_FOUNDRY}.cognitiveservices.azure.com",
    f"{STD_FOUNDRY}.openai.azure.com",
    f"{STD_STORAGE}.blob.core.windows.net",
    f"{COSMOS}.documents.azure.com",
    f"{SEARCH}.search.windows.net",
]
resolution_rows = []
for name in names:
    addresses = sorted({item[4][0] for item in socket.getaddrinfo(name, 443, type=socket.SOCK_STREAM)})
    private = all(ipaddress.ip_address(address) in private_endpoint_network for address in addresses)
    resolution_rows.append({"name": name, "addresses": ", ".join(addresses), "in_pe_subnet": private})
display(pd.DataFrame(resolution_rows))
if not all(row["in_pe_subnet"] for row in resolution_rows):
    raise RuntimeError("Private DNS validation failed. Stop and involve Zava networking/DNS.")

## 10. Create capability hosts exactly once

Both host lists must be empty before initial deployment. If one host already exists, do not deploy the combined template; use the safe recovery cell.

In [ ]:
account_hosts_url = f"https://management.azure.com{STD_ID}/capabilityHosts?api-version=2025-06-01"
project_hosts_url = f"https://management.azure.com{STD_ID}/projects/{STD_PROJECT}/capabilityHosts?api-version=2025-06-01"
account_hosts = az_json("rest", "--method", "GET", "--url", account_hosts_url)["value"]
project_hosts = az_json("rest", "--method", "GET", "--url", project_hosts_url)["value"]
display(pd.DataFrame([{"scope": "account", "name": h["name"], "state": h["properties"].get("provisioningState")} for h in account_hosts] +
                     [{"scope": "project", "name": h["name"], "state": h["properties"].get("provisioningState")} for h in project_hosts]))
if account_hosts or project_hosts:
    raise RuntimeError("A capability host already exists. Do not deploy the combined host template.")

In [ ]:
show_source(HOST_TEMPLATE)
build_bicep(HOST_TEMPLATE)
HOST_DEPLOYMENT = f"zava-05-capability-hosts-{ENVIRONMENT}"
host_parameters = [
    f"workloadName={WORKLOAD_NAME}", f"environment={ENVIRONMENT}",
    f"networkResourceGroupName={NETWORK_RG}", f"vnetName={VNET_NAME}",
    f"agentSubnetName={AGENT_SUBNET_NAME}",
]
az("deployment", "group", "validate", "-g", WORKLOAD_RG, "-n", f"{HOST_DEPLOYMENT}-validate",
   "-f", str(HOST_TEMPLATE), "-p", *host_parameters, "-o", "none")
az("deployment", "group", "what-if", "-g", WORKLOAD_RG, "-n", HOST_DEPLOYMENT,
   "-f", str(HOST_TEMPLATE), "-p", *host_parameters, "--result-format", "ResourceIdOnly")

In [ ]:
APPROVE_ONE_TIME_CAPABILITY_HOSTS = False
require_approval(APPROVE_ONE_TIME_CAPABILITY_HOSTS, "One-time capability-host deployment")
az("deployment", "group", "create", "-g", WORKLOAD_RG, "-n", HOST_DEPLOYMENT,
   "-f", str(HOST_TEMPLATE), "-p", *host_parameters, "-o", "none")
print("Capability-host deployment submitted.")

In [ ]:
account_hosts = az_json("rest", "--method", "GET", "--url", account_hosts_url)["value"]
project_hosts = az_json("rest", "--method", "GET", "--url", project_hosts_url)["value"]
rows = [{"scope": "account", "name": h["name"], "state": h["properties"].get("provisioningState")} for h in account_hosts]
rows += [{"scope": "project", "name": h["name"], "state": h["properties"].get("provisioningState"),
          "storage": h["properties"].get("storageConnections"),
          "threads": h["properties"].get("threadStorageConnections"),
          "vectors": h["properties"].get("vectorStoreConnections")} for h in project_hosts]
display(pd.DataFrame(rows))
if len(account_hosts) != 1 or len(project_hosts) != 1 or any(h["properties"].get("provisioningState") != "Succeeded" for h in account_hosts + project_hosts):
    raise RuntimeError("Both capability hosts must exist and report Succeeded.")

In [ ]:
# RECOVERY ONLY — creates missing hosts, never overwrites an existing host.
APPROVE_MISSING_HOST_RECOVERY = False
require_approval(APPROVE_MISSING_HOST_RECOVERY, "Missing capability-host recovery")

account_hosts = az_json("rest", "--method", "GET", "--url", account_hosts_url)["value"]
project_hosts = az_json("rest", "--method", "GET", "--url", project_hosts_url)["value"]

def put_json(url, body):
    with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False, encoding="utf-8") as handle:
        json.dump(body, handle)
        temp_path = handle.name
    try:
        az("rest", "--method", "PUT", "--url", url, "--headers", "Content-Type=application/json", "--body", f"@{temp_path}")
    finally:
        Path(temp_path).unlink(missing_ok=True)

if not account_hosts:
    put_json(
        f"https://management.azure.com{STD_ID}/capabilityHosts/{STD_FOUNDRY}-cap?api-version=2025-06-01",
        {"properties": {"capabilityHostKind": "Agents", "customerSubnet": AGENT_SUBNET_ID}},
    )
    raise RuntimeError("Account host was created. Wait until it is Succeeded, then rerun this recovery cell for a missing project host.")

account_state = account_hosts[0]["properties"].get("provisioningState")
if account_state != "Succeeded":
    raise RuntimeError(f"Account host exists in {account_state} state; do not overwrite it.")

if not project_hosts:
    put_json(
        f"https://management.azure.com{STD_ID}/projects/{STD_PROJECT}/capabilityHosts/{STD_PROJECT}-cap?api-version=2025-06-01",
        {"properties": {
            "capabilityHostKind": "Agents",
            "storageConnections": ["agent-storage"],
            "threadStorageConnections": ["agent-thread-storage"],
            "vectorStoreConnections": ["agent-vector-store"],
        }},
    )
else:
    print("Project host already exists; no recovery action taken.")

## 11. Deploy the approved model

Only continue after both capability hosts report `Succeeded` and Zava confirms model availability and quota.

In [ ]:
MODEL_DEPLOYMENT = "<approved-deployment-name>"
MODEL_NAME = "<approved-model-name>"
MODEL_VERSION = "<approved-model-version>"
MODEL_CAPACITY = 0

for name, value in {"MODEL_DEPLOYMENT": MODEL_DEPLOYMENT, "MODEL_NAME": MODEL_NAME, "MODEL_VERSION": MODEL_VERSION}.items():
    if not value or (value.startswith("<") and value.endswith(">")):
        raise ValueError(f"Configure {name}.")
if MODEL_CAPACITY < 1:
    raise ValueError("Set MODEL_CAPACITY to an approved positive value.")
print(json.dumps({"account": STD_FOUNDRY, "deployment": MODEL_DEPLOYMENT, "model": MODEL_NAME,
                  "version": MODEL_VERSION, "sku": "GlobalStandard", "capacity": MODEL_CAPACITY}, indent=2))

In [ ]:
APPROVE_MODEL_DEPLOYMENT = False
require_approval(APPROVE_MODEL_DEPLOYMENT, "Model deployment")
az("cognitiveservices", "account", "deployment", "create", "-g", WORKLOAD_RG, "-n", STD_FOUNDRY,
   "--deployment-name", MODEL_DEPLOYMENT, "--model-name", MODEL_NAME,
   "--model-version", MODEL_VERSION, "--model-format", "OpenAI",
   "--sku-name", "GlobalStandard", "--sku-capacity", str(MODEL_CAPACITY), "-o", "none")
az("cognitiveservices", "account", "deployment", "list", "-g", WORKLOAD_RG, "-n", STD_FOUNDRY,
   "--query", "[].{name:name,model:properties.model.name,version:properties.model.version,state:properties.provisioningState,sku:sku.name,capacity:sku.capacity}", "-o", "table")

## 12. Inspect resources and export non-secret outputs

This writes deployment outputs and resource inventory to a local JSON file without credentials or secrets.

In [ ]:
resources = list(resource_client.resources.list_by_resource_group(WORKLOAD_RG))
resource_rows = [{
    "name": r.name, "type": r.type, "location": r.location,
    "id": r.id,
} for r in resources]
display(pd.DataFrame(resource_rows).sort_values(["type", "name"]))

export_data = {
    "subscription_id": SUBSCRIPTION_ID,
    "tenant_id": TENANT_ID,
    "workload_resource_group": WORKLOAD_RG,
    "network": {"vnet": VNET_NAME, "agent_subnet_id": AGENT_SUBNET_ID, "private_endpoint_subnet": PE_SUBNET_NAME},
    "chapter01": chapter01_outputs,
    "chapter01a": core_outputs,
    "resources": resource_rows,
}
EXPORT_PATH = INFRA_DIR / f"zava-{ENVIRONMENT}-deployment-outputs.json"
EXPORT_PATH.write_text(json.dumps(export_data, indent=2), encoding="utf-8")
print(f"Exported non-secret deployment data to {EXPORT_PATH}")

## 13. Optional cleanup — dedicated workload resource group only

**Never run this for a shared resource group or a resource group containing Zava-owned VNet/DNS resources.** Deleting a resource group is irreversible and removes everything in it. Foundry network injection may also require deleting and purging Foundry resources before network cleanup.

In [ ]:
APPROVE_RESOURCE_GROUP_DELETE = False
TYPE_RESOURCE_GROUP_NAME_TO_CONFIRM = ""

if not APPROVE_RESOURCE_GROUP_DELETE:
    raise RuntimeError("Cleanup blocked. Set APPROVE_RESOURCE_GROUP_DELETE=True only for a dedicated disposable workload RG.")
if TYPE_RESOURCE_GROUP_NAME_TO_CONFIRM != WORKLOAD_RG:
    raise RuntimeError("Cleanup blocked. TYPE_RESOURCE_GROUP_NAME_TO_CONFIRM must exactly equal WORKLOAD_RG.")
if WORKLOAD_RG in {NETWORK_RG, DNS_RG}:
    raise RuntimeError("Cleanup blocked: workload RG also contains customer network or DNS resources.")

poller = resource_client.resource_groups.begin_delete(WORKLOAD_RG)
print(f"Deletion started for dedicated workload resource group: {WORKLOAD_RG}")
# To wait synchronously, explicitly run: poller.result()

# Final acceptance checklist

- [ ] Zava confirmed `dmzsubnet-1` is `10.75.139.128/29`.
- [ ] Existing DMZ and hybrid subnets are unchanged.
- [ ] Agent subnet is `/27`, exclusive, and delegated to `Microsoft.App/environments`.
- [ ] Private endpoint subnet is `/28` and contains only approved endpoints.
- [ ] All seven private endpoints are approved.
- [ ] Customer-owned DNS resolves all service FQDNs privately.
- [ ] Public access and local/shared-key authentication are disabled where supported.
- [ ] Standard account network injection references the approved agent subnet.
- [ ] Project identity roles and all three AAD connections exist.
- [ ] Account and project capability hosts report `Succeeded`.
- [ ] Approved model deployment reports `Succeeded`.
- [ ] A temporary Standard-project agent completes a private-path model test.